# 07. Tinier-YOLO cu cameră USB — captură la cerere

Folosiți o **cameră USB conectată direct la placa PYNQ** (de exemplu Logitech C270), nu camera laptopului. Notebook-ul reutilizează modelul Tinier-YOLO instalat din [laboratorul 05](05_tiny-yolo-image.ipynb), cu 20 de clase VOC, printre care persoană, automobil, bicicletă, câine, sticlă și scaun.

![Topologia Tinier-YOLO](../Tinier-YOLO-topology.svg)

Executați celulele în ordine cu **Shift + Enter** și opriți celelalte kerneluri FPGA/cameră. Pachetul `qnn` instalat trebuie să aibă importurile `Xlnk` neutilizate eliminate. Nu sunt necesare alte modele, compilarea unui bitstream, reinstalarea sau antrenarea.

La fiecare apăsare de buton se captează **o imagine statică nouă**, după care camera este eliberată și imaginea este procesată. Nu există un fir de execuție permanent pentru cameră. Convoluțiile CPU pot fi lente; nu este o demonstrație video fluidă, în timp real. Rezultatele anterioare rămân vizibile, fără a fi salvate în fișiere separate.

**Stocare:** nu sunt salvate foldere cu rezultate, PNG-uri, scoruri sau jurnale JSON. Biblioteca nativă utilizează fișiere temporare mici în `/dev/shm` (RAM), eliminate după procesare și la finalul sesiunii. Istoricul păstrează previzualizări reduse în memorie până la repornirea kernelului. Jupyter poate include imaginile afișate la salvarea notebook-ului; utilizați **Cell → All Output → Clear** înainte de salvare. Fișierele originale ale modelului rămân nemodificate.

*Copie tradusă a laboratorului didactic; pentru funcționarea din subdirector, se verifică și directorul părinte al resurselor. Această copie nu a fost validată separat pe hardware.*

## 1. Bibliotecile, modelul și configurarea camerei

Executați celula. Sunt identificate modelul existent și instalarea Darknet personalizată și sunt enumerate dispozitivele `/dev/video*`. Implicit se folosește `/dev/video0`, solicitând 640×480. Pentru alt dispozitiv, modificați `CAMERA_DEVICE` după verificarea listei. Dacă este aleasă altă rezoluție de cameră, se folosesc dimensiunile efectiv returnate.

Fișierele lipsă sunt raportate înainte de inițializarea FPGA. `ipywidgets` furnizează butoanele, dacă este disponibil, dar se pot apela funcțiile și manual. Pachetele native OpenCV/Darknet sunt cele deja folosite pe placă.

In [ ]:
import sys
import os
import json
import io
import traceback
from collections import Counter
from contextlib import nullcontext
import cv2
import gc
import random
import time
import ctypes
import tempfile
import os
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageFont
from matplotlib import pyplot as plt
from IPython.display import display
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

import qnn
from qnn import TinierYolo, utils

QNN_PARAM_DIR = Path(qnn.__file__).resolve().parent / "params"
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "Tinier-YOLO-topology.svg").is_file():
    if (NOTEBOOK_DIR.parent / "Tinier-YOLO-topology.svg").is_file():
        NOTEBOOK_DIR = NOTEBOOK_DIR.parent
    else:
        NOTEBOOK_DIR = Path("/home/xilinx/jupyter_notebooks/qnn")
IMAGE_DIR = NOTEBOOK_DIR / "yoloimages"
DARKNET_DIR = Path("/opt/darknet")
VOC_NAMES = DARKNET_DIR / "data/voc.names"
required_files = [QNN_PARAM_DIR / name for name in (
    "tinier-yolo-layers.json", "tinier-yolo-conv0-W.npy", "tinier-yolo-conv8-W.npy",
    "tinier-yolo-conv0-bias.npy", "tinier-yolo-conv8-bias.npy",
    "tinier-yolo-bwn-3bit-relu-nomaxpool.cfg")]
required_files += [DARKNET_DIR / "python/darknet.py", DARKNET_DIR / "libdarknet.so", VOC_NAMES]
# Darknet loads all printable-character PNGs at each size when drawing labels.
required_files += [DARKNET_DIR / "data/labels" / "{}_{}.png".format(char, size)
                   for size in range(8) for char in range(32, 127)]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Required files missing (first 12):\n" + "\n".join(missing[:12]))
if str(DARKNET_DIR / "python") not in sys.path:
    sys.path.insert(0, str(DARKNET_DIR / "python"))
import darknet as dn

dn.lib.parse_network_cfg.argtypes = [ctypes.c_char_p]
dn.lib.parse_network_cfg.restype = ctypes.c_void_p
dn.lib.forward_region_layer_pointer_nolayer.argtypes = [ctypes.c_void_p, ctypes.POINTER(ctypes.c_float)]
dn.lib.forward_region_layer_pointer_nolayer.restype = None
dn.lib.draw_detection_python.argtypes = [ctypes.c_void_p, ctypes.c_char_p,
    ctypes.c_float, ctypes.c_float, ctypes.c_char_p, ctypes.c_char_p, ctypes.c_char_p, ctypes.c_char_p]
dn.lib.draw_detection_python.restype = None
dn.lib.free_network.argtypes = [ctypes.c_void_p]
dn.lib.free_network.restype = None
print("Parameters:", QNN_PARAM_DIR)
print("USB devices:", ", ".join(str(p) for p in sorted(Path("/dev").glob("video*"))) or "none")

CAMERA_DEVICE = "/dev/video0"
CAMERA_WIDTH, CAMERA_HEIGHT = 640, 480
WARMUP_FRAMES = 12
THRESHOLD = 0.3
CAPTURE_INDEX = 0
# Close controls from an earlier run before replacing model state.
for control in globals().get("ui_buttons", []):
    control.disabled = True
if globals().get("ui_panel") is not None:
    ui_panel.close()
print("Widgets:", "available" if widgets is not None else "not installed; use the manual call below")

def memory_preview(value):
    if isinstance(value, Image.Image):
        image = value.copy()
    else:
        with Image.open(str(value)) as opened:
            image = opened.convert("RGB")
    image.thumbnail((480, 360))
    return image

def clear_intermediate_files():
    directory = globals().get("RESULT_DIR")
    if directory is not None and directory.is_dir():
        for path in directory.iterdir():
            if path.is_file():
                path.unlink()



## 2. Încărcarea modelului o singură dată

Așteptați **Model ready**. Prima și ultima convoluție rulează pe CPU ARM, straturile cuantizate intermediare pe FPGA, iar Darknet realizează transformarea letterbox. Ponderile mari nu se descarcă din nou. Reexecutarea secțiunii începe un istoric nou în memorie; închideți comenzile anterioare cu **Finish / release**.

In [ ]:
def release_model():
    global classifier, net_darknet
    try:
        if globals().get("classifier") is not None:
            classifier.deinit_accelerator()
    finally:
        classifier = None
        if globals().get("net_darknet"):
            dn.lib.free_network(net_darknet)
        net_darknet = None
        if globals().get("_result_workspace") is not None:
            _result_workspace.cleanup()

release_model()
for variable in ("conv0_weights", "conv8_weights", "conv0_bias", "conv8_bias"):
    globals().pop(variable, None)
gc.collect()
print("Loading weights...", flush=True)
conv0_weights = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv0-W.npy"),
                        encoding="latin1", allow_pickle=True).transpose(3, 2, 1, 0)
conv8_weights = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv8-W.npy"),
                        encoding="latin1", allow_pickle=True).transpose(3, 2, 1, 0)
conv0_bias = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv0-bias.npy"),
                     encoding="latin1", allow_pickle=True).reshape(-1, 1)
conv8_bias = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv8-bias.npy"),
                     encoding="latin1", allow_pickle=True).reshape(-1, 1)
classifier = None
net_darknet = None
if globals().get("_result_workspace") is not None:
    _result_workspace.cleanup()
if not Path("/dev/shm").is_dir():
    raise RuntimeError("RAM-backed /dev/shm is required; no SD-card fallback is used.")
_result_workspace = tempfile.TemporaryDirectory(prefix="qnn-07-camera-", dir="/dev/shm")
RESULT_DIR = Path(_result_workspace.name)
try:
    classifier = TinierYolo()
    net = classifier.load_network()
    out_ch, out_dim, _ = net["conv7"]["output"]
    first_ch, first_dim, _ = net["conv1"]["input"]
    if conv0_weights.shape[0] != first_ch or conv0_bias.shape != (first_ch, 1):
        raise ValueError("First-convolution weights do not match the network.")
    if conv8_weights.shape[:2] != (125, out_ch) or conv8_bias.shape != (125, 1):
        raise ValueError("Expected the supplied 20-class, 5-anchor VOC detection head.")
    conv0_bias = np.broadcast_to(conv0_bias, (first_ch, first_dim * first_dim))
    conv8_bias = np.broadcast_to(conv8_bias, (125, out_dim * out_dim))
    net_darknet = dn.lib.parse_network_cfg(str(
        QNN_PARAM_DIR / "tinier-yolo-bwn-3bit-relu-nomaxpool.cfg").encode())
    if not net_darknet:
        raise RuntimeError("Darknet failed to load its network configuration.")
    print("Initializing FPGA...", flush=True)
    classifier.init_accelerator()
except BaseException:
    release_model()
    raise
# Only temporary native-library files are placed in RAM.
RESULT_INDEX = 0
detection_history = []
print("Model ready. Intermediate files use RAM and are removed after processing.")

CAPTURE_INDEX = 0


## 3. Definirea funcțiilor pentru model și cameră

Executați celula. Aceasta definește etapele modelului și captarea sigură a camerei. La fiecare captură, camera USB este stabilizată și apoi eliberată în `finally`, **înainte** de inferența lentă. Biblioteca nativă veche folosește un PNG temporar în RAM, citit cu aceeași preprocesare ca în laboratorul 05. Această celulă nu deschide camera.

In [ ]:
def load_detector_image(image_path):
    original = None
    letterboxed = None
    try:
        original = dn.load_image(str(image_path).encode(), 0, 0)
        if not original.data:
            raise RuntimeError("Darknet could not read " + str(image_path))
        letterboxed = dn.letterbox_image(original, 416, 416)
        if not letterboxed.data:
            raise RuntimeError("Darknet letterboxing failed.")
        pixels = np.copy(np.ctypeslib.as_array(letterboxed.data, (3, 416, 416)))
        # Preserve the original trained model's paired spatial-axis swaps.
        return np.swapaxes(pixels, 0, 2)[np.newaxis, :, :, :]
    finally:
        if letterboxed is not None and letterboxed.data:
            dn.free_image(letterboxed)
        if original is not None and original.data:
            dn.free_image(original)

def first_convolution(pixels):
    if pixels.shape != (1, 416, 416, 3):
        raise ValueError("Expected one letterboxed 416x416 RGB image.")
    values = utils.conv_layer(pixels, conv0_weights, b=conv0_bias, stride=2, padding=1)
    return utils.quantize(values.clip(0.0, 4.0) / 4.0, 3)

def fpga_layers(quantized):
    if classifier is None:
        raise RuntimeError("Model released. Rerun section 2 before inference.")
    output = classifier.get_accel_buffer(out_ch, out_dim)
    inputs = np.ascontiguousarray(classifier.prepare_buffer(quantized * 7))
    started = time.perf_counter()
    classifier.inference(inputs, output)
    call_seconds = time.perf_counter() - started
    values = classifier.postprocess_buffer(output)
    if values.size != out_ch * out_dim * out_dim or not np.all(np.isfinite(values)):
        raise ValueError("Invalid FPGA output.")
    return values, call_seconds

def last_convolution(values):
    pixels = np.swapaxes(values.reshape(out_dim, out_dim, out_ch), 0, 1)[np.newaxis]
    output = utils.conv_layer(pixels, conv8_weights, b=conv8_bias, stride=1)
    # The native region layer reads float*, so float32 and contiguous storage are required.
    output = np.ascontiguousarray(output, dtype=np.float32)
    if output.shape != (125, out_dim, out_dim) or not np.all(np.isfinite(output)):
        raise ValueError("Invalid detection-head output.")
    return output

def draw_detections(image_path, values, threshold=0.3):
    global RESULT_INDEX
    if net_darknet is None:
        raise RuntimeError("Model released. Rerun section 2.")
    if not 0 < threshold < 1:
        raise ValueError("Detection threshold must be between 0 and 1.")
    # Defensive check also protects callers that rerun this cell independently.
    if values.dtype != np.float32 or not values.flags.c_contiguous:
        raise ValueError("Darknet input must be contiguous float32.")
    if values.shape != (125, out_dim, out_dim) or not np.all(np.isfinite(values)):
        raise ValueError("Wrong detection-head shape or non-finite values.")
    RESULT_INDEX += 1
    result_name = "detection_{:04d}".format(RESULT_INDEX)
    output_prefix = RESULT_DIR / result_name
    output_image = RESULT_DIR / (result_name + ".png")
    probabilities = RESULT_DIR / (result_name + "_probabilities.txt")
    # This legacy C helper uses 256-byte path buffers, including added suffixes.
    paths = [str(image_path), str(VOC_NAMES), str(DARKNET_DIR) + "/",
             str(output_prefix), str(probabilities)]
    if any(len(path.encode()) > 240 for path in paths):
        raise ValueError("Use a shorter notebook/image path for the legacy Darknet helper.")
    for path in (output_image, probabilities):
        if path.exists():
            path.unlink()
    try:
        dn.lib.forward_region_layer_pointer_nolayer(
            net_darknet, values.ctypes.data_as(ctypes.POINTER(ctypes.c_float)))
        dn.lib.draw_detection_python(net_darknet, paths[0].encode(), ctypes.c_float(threshold),
            ctypes.c_float(0.5), paths[1].encode(), paths[2].encode(),
            paths[3].encode(), paths[4].encode())
        if not output_image.is_file() or not probabilities.is_file():
            raise RuntimeError("Darknet did not create the detection outputs.")
        detections = []
        for line in probabilities.read_text().splitlines():
            if not line.strip():
                continue
            name, score = line.rsplit(":", 1)
            detections.append((name.strip(), float(score.strip().rstrip("%")) / 100.0))
        return memory_preview(output_image), sorted(detections, key=lambda item: item[1], reverse=True)
    finally:
        for path in (output_image, probabilities):
            if path.exists():
                path.unlink()


def show_detections(output_image, detections):
    for name, score in detections:
        print("{:20s} {:6.0%}".format(name, score))
    if not detections:
        print("No objects above the selected threshold. This is a valid result.")
    preview = memory_preview(output_image)
    preview.thumbnail((800, 600))
    display(preview)

def capture_frame():
    """Open a board-connected USB camera, get a fresh frame, always close it."""
    cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
    try:
        if not cap.isOpened():
            raise RuntimeError("Cannot open {}. Connect the USB camera to the board and close other camera kernels.".format(CAMERA_DEVICE))
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_WIDTH)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_HEIGHT)
        for _ in range(WARMUP_FRAMES):
            ok, frame = cap.read()
            if not ok or frame is None or frame.size == 0:
                raise RuntimeError("No camera frame. Check USB power/connection and camera ownership.")
        return frame.copy()  # OpenCV BGR; cv2.imwrite preserves the correct PNG colors.
    finally:
        cap.release()

def save_capture(frame):
    global CAPTURE_INDEX
    CAPTURE_INDEX += 1
    path = RESULT_DIR / "capture_{:04d}.png".format(CAPTURE_INDEX)
    if not cv2.imwrite(str(path), frame):
        raise IOError("Could not save camera capture: " + str(path))
    return path

def preview_camera():
    frame = capture_frame()
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    preview = Image.fromarray(rgb)
    preview.thumbnail((640, 480))
    print("Camera preview — captured now; the camera has been released.")
    display(preview)
    return frame

def run_model(image_path):
    if classifier is None:
        raise RuntimeError("Session finished. Rerun section 2 and the controls cell.")
    if not classifier.init:
        print("Loading YOLO overlay...", flush=True)
        classifier.init_accelerator()
    timings = {}
    started = time.perf_counter()
    pixels = load_detector_image(image_path)
    timings["letterbox_s"] = time.perf_counter() - started
    started = time.perf_counter()
    quantized = first_convolution(pixels)
    timings["first_cpu_s"] = time.perf_counter() - started
    started = time.perf_counter()
    fpga_output, timings["fpga_call_s"] = fpga_layers(quantized)
    timings["fpga_with_buffers_s"] = time.perf_counter() - started
    started = time.perf_counter()
    head = last_convolution(fpga_output)
    timings["last_cpu_s"] = time.perf_counter() - started
    return head, timings

def show_pair(original_path, result_path):
    original = memory_preview(original_path)
    annotated = memory_preview(result_path)
    original.thumbnail((480, 360))
    annotated.thumbnail((480, 360))
    canvas = Image.new("RGB", (original.width + annotated.width, max(original.height, annotated.height) + 28), "white")
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 7), "Captured frame", fill="black")
    draw.text((original.width + 8, 7), "Detection result", fill="black")
    canvas.paste(original, (0, 28))
    canvas.paste(annotated, (original.width, 28))
    display(canvas)


## 4. Definirea capturii și detecției

Executați celula pentru a defini acțiunea butonului; încă nu se captează nimic. Detecția desenează dreptunghiuri și afișează aceleași scoruri ca exemplul pe imagine. Scorul exprimă încrederea, nu acuratețea. Un rezultat fără detecții este valid. Măsurătorile includ deschiderea/stabilizarea camerei, fișierele imagine și procesarea modelului, cu timpul apelului FPGA raportat separat. Nu includ afișarea Jupyter și nu reprezintă FPS al unui flux video.

In [ ]:
def capture_and_detect():
    if classifier is None:
        raise RuntimeError("Session finished. Rerun section 2, then the controls cell.")
    if not 0 < THRESHOLD < 1:
        raise ValueError("THRESHOLD must be between 0 and 1.")
    started = time.perf_counter()
    print("Capturing USB camera...", flush=True)
    try:
        frame = capture_frame()
        capture_path = save_capture(frame)
        print("Captured", capture_path.name, "— processing; please wait...", flush=True)
        head, timings = run_model(capture_path)
        post_started = time.perf_counter()
        result_path, detections = draw_detections(capture_path, head, THRESHOLD)
        timings["decode_draw_s"] = time.perf_counter() - post_started
        timings["capture_to_result_s"] = time.perf_counter() - started
        record = dict(capture=memory_preview(capture_path), result=memory_preview(result_path),
                      threshold=THRESHOLD, detections=detections, timings=timings)
        detection_history.append(record)
        print("Result {} — {}".format(len(detection_history), capture_path.name))
        show_pair(capture_path, result_path)
        for name, score in detections:
            print("{:20s} {:6.0%}".format(name, score))
        if not detections:
            print("No objects above threshold. Try another scene, distance or lighting.")
        print("Capture to result: {:.3f} s; FPGA inference call: {:.3f} s".format(
            timings["capture_to_result_s"], timings["fpga_call_s"]))
        print("Includes camera opening/warmup, model stages, drawing and PNG I/O; excludes notebook rendering.")
        return record
    except BaseException:
        classifier.deinit_accelerator()
        raise
    finally:
        clear_intermediate_files()


def finish_session():
    for control in globals().get("ui_buttons", []):
        control.disabled = True
    release_model()
    print("FPGA runtime and Darknet released. The camera is closed after every capture.")


## 5. Experimente propuse studenților

Folosiți o persoană, o sticlă sau un scaun, toate clase acceptate. Comparați trei distanțe și două condiții de iluminare. Încercați ocluzia parțială. Modificați `THRESHOLD` într-o celulă nouă (de exemplu `THRESHOLD = 0.5`) și capturați din nou; comparați dreptunghiurile păstrate și scorurile. Nu determinați acuratețea unui set de date din doar câteva capturi. Comparați timpii etapelor CPU și apelului FPGA.

## 6. Utilizarea camerei

Executați o singură dată celula cu butoanele, apoi folosiți comenzile:

1. **Preview camera** realizează o previzualizare statică pentru orientarea camerei. Nu face inferență și nu este un flux video.
2. **Capture + detect** captează o imagine nouă la apăsare și o procesează. Păstrați obiectul în cadru în momentul capturii, apoi așteptați **Ready**. Imaginea originală și rezultatul apar alăturat, urmate de scoruri și durate. Scena poate să se fi schimbat față de previzualizare.
3. Repetați după necesități. Imaginile se adaugă în istoric. **Finish / release** dezactivează controalele și eliberează modelul.

Butoanele sunt dezactivate în timpul procesării pentru a evita suprapunerea apelurilor cameră/FPGA. În modul manual, rulați `preview_camera()` sau `capture_and_detect()` într-o celulă nouă. Reexecutarea unei celule Jupyter înlocuiește afișarea ei, dar istoricul în memorie rămâne până la repornirea kernelului.

In [ ]:
# Close old controls so a rerun cannot leave a second set of active buttons.
for control in globals().get("ui_buttons", []):
    control.disabled = True
if globals().get("ui_panel") is not None:
    ui_panel.close()

if widgets is not None:
    preview_button = widgets.Button(description="Preview camera", icon="camera")
    detect_button = widgets.Button(description="Capture + detect", icon="search", button_style="primary")
    finish_button = widgets.Button(description="Finish / release", icon="stop", button_style="warning")
    status = widgets.HTML("Ready. Aim the camera, optionally preview, then capture and detect.")
    history_output = widgets.Output()
    ui_buttons = [preview_button, detect_button, finish_button]
    ui_panel = widgets.VBox([widgets.HBox(ui_buttons), status, history_output])

    def perform(action):
        for control in ui_buttons:
            control.disabled = True
        status.value = "Working — wait for the current capture/inference to finish."
        try:
            with history_output:
                action()
            status.value = "Ready. Previous images remain below."
        except Exception:
            with history_output:
                traceback.print_exc()
            status.value = "Operation failed. See the message below; previous results are retained."
        finally:
            for control in ui_buttons:
                control.disabled = classifier is None

    def finish_clicked(_):
        for control in ui_buttons:
            control.disabled = True
        finish_session()
        status.value = "Session released. Rerun section 2 and this cell to start again."

    preview_button.on_click(lambda _: perform(preview_camera))
    detect_button.on_click(lambda _: perform(capture_and_detect))
    finish_button.on_click(finish_clicked)
    display(ui_panel)
else:
    ui_buttons = []
    ui_panel = None
    print("Manual mode: run preview_camera() or capture_and_detect() in a new cell.")


## 7. Reafișarea istoricului în memorie (opțional)

Executați celula pentru a vedea toate perechile imagine/detecție finalizate, inclusiv cele rezultate din apelurile manuale. Citește previzualizările din memorie; **nu** efectuează inferență din nou.

In [ ]:
print("Completed detections:", len(detection_history))
for number, record in enumerate(detection_history, 1):
    print("Result", number)
    show_pair(record["capture"], record["result"])



## 8. Finalizare și eliberarea memoriei

Folosiți **Finish / release** sau executați celula următoare. Ambele variante pot fi repetate în siguranță. Închideți acest kernel înaintea altui notebook FPGA. Pentru reluare, executați secțiunea 2 și celula controalelor; reluați și configurarea dacă ați schimbat valorile.

În cazul unui apel nativ lung, **Kernel → Interrupt** poate avea efect doar când apelul revine. Nu porniți un al doilea notebook FPGA în paralel.

In [ ]:
finish_session()
for variable in ("conv0_weights", "conv8_weights", "conv0_bias", "conv8_bias"):
    globals().pop(variable, None)
gc.collect()
print("Large weights cleared. No capture or result files were saved.")


Referințe de implementare: [OpenCV VideoCapture](https://docs.opencv.org/3.4.20/d8/dfe/classcv_1_1VideoCapture.html), [QNN-MO-PYNQ](https://github.com/onigas/QNN-MO-PYNQ), [stratul region din Darknet](https://github.com/giuliogamba/darknet/blob/master/src/region_layer.c) și [overlay-ul de bază PYNQ-Z2](https://pynq.readthedocs.io/en/v2.5/pynq_overlays/pynqz2/pynqz2_base_overlay.html).

**Validare:** sintaxa notebook-ului, gestionarea camerei/FPGA, reinițializarea, istoricul în memorie, callback-urile și fluxul Python au fost verificate off-board cu interfețe hardware/native simulate. Camera USB reală, widgeturile, Darknet, timpii și predicțiile FPGA și comportamentul LED-ului fizic necesită încă verificare pe placa PYNQ.